# Implementing Calibration and Misbehavior Detection

In [1]:
import sys
import cv2
import numpy as np
import pandas as pd
import logging
import os
import time
sys.path.append('../src')  # Adjust path depending on your structure

print("Current Directory:", os.getcwd())

from head_pose_estimator import HeadPoseEstimator
from mediapipe_face_mesh_detector import MediapipeFaceMeshDetector
from read_video_file import get_video_frames
from mouth_state_detector import MouthStateDetection


Current Directory: /Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/notebooks


## Step1: Define Calibration Procedure
Calibration Phase:
- Ask the user to look directly at the screen where typical content would be displayed (e.g. center of the screen where exam questions are likely to appear).
- Capture the head pose over a few seconds to average out any minor movements and determine a baseline "neutral" head pose.

In [6]:

HEAD_POSE_ESTIMATOR_MODEL = "../src/models/head_pose_model/LGBM_finetune_v_optuna.pkl"
face_mesh_detector = MediapipeFaceMeshDetector()
mouth_state_detector = MouthStateDetection()

print("...Loaded models")

...Loaded models


## Test with subjects

### Subject 1

In [8]:
def detect_mouth_state_behavior(video_path, threshold=0.01, output_path=None):
    mouth_state_detector = MouthStateDetection(talking_threshold=threshold)
    print("Detecting mouth state behavior...")
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    print("FPS:", fps)
    current_frame = 0
    results = []
    
    os.makedirs(output_path, exist_ok=True)
    
    for frame in get_video_frames(video_path):
        if frame is not None:
            frame = cv2.flip(frame, 1)
            
            mediapipe_results = face_mesh_detector.detect_landmarks(frame)
            
            output = mouth_state_detector.extract_mouth_state(mediapipe_results)
            
            
            
            # Display the mouth state on the frame
            cv2.putText(frame, output["classify"], (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(frame, f'Mouth Opening Distance: {output["mouth_opening_distance"]:.2f}', (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(frame, f"Time: {time_MMSS}", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            cv2.imshow("Frame", frame)
            
            if cv2.waitKey(25) == 27:  # Exit on pressing the ESC key
                break
            
            if current_frame % fps == 0:
                time_MMSS = time.strftime("%M:%S", time.gmtime(current_frame // fps))
                output["time"] = time_MMSS
                results.append(output)
                cv2.imwrite(f"{output_path}/{current_frame}_{time_MMSS}.jpg", frame)
            
            current_frame += 1
            
        else:
            break
    cv2.destroyAllWindows()

    df = pd.DataFrame(results)
    df.to_csv(f"{output_path}/mouth_state_results.csv", index=False)

In [9]:
def detect_mouth_state_behavior_result(video_path, threshold=0.01, output_path=None):
    mouth_state_detector = MouthStateDetection(talking_threshold=threshold)
    print("Detecting mouth state behavior...")
    
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    print("FPS:", fps)
    current_frame = 0
    results = []
    
    os.makedirs(output_path, exist_ok=True)
    
    for frame in get_video_frames(video_path):
        if frame is not None:
            # Process every second of the video
            if current_frame % fps == 0:
                frame = cv2.flip(frame, 1)
                mediapipe_results = face_mesh_detector.detect_landmarks(frame)
                output = mouth_state_detector.extract_mouth_state(mediapipe_results)
                time_MMSS = time.strftime("%M:%S", time.gmtime(current_frame // fps))
                output["time"] = time_MMSS
                results.append(output)
                
                # Display the mouth state on the frame
                cv2.putText(frame, output["classify"], (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                cv2.putText(frame, f'Mouth Opening Distance: {output["mouth_opening_distance"]:.2f}', (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                cv2.putText(frame, f"Time: {time_MMSS}", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                
                cv2.imwrite(f"{output_path}/{current_frame}_{time_MMSS}.jpg", frame)
            
            current_frame += 1
            
        else:
            break
    cv2.destroyAllWindows()
    
    df = pd.DataFrame(results)
    df.to_csv(f"{output_path}/mouth_state_results.csv", index=False)

In [20]:
SUBJECT_VIDEO_PATH = "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/gritt_exam.mov"

In [21]:
detect_mouth_state_behavior_result(SUBJECT_VIDEO_PATH, 0.01, "mouth/gritt_exam")

Detecting mouth state behavior...
FPS: 30.0


### Subject 3

In [34]:
SUBJECT_VIDEO_PATH = "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/processed/OEP_dataset/subject3/XiYin1.mp4"

In [20]:
SUBJECT_VIDEO_PATH = "/Users/sarmkunatham/Desktop/1. Projects/GithubCodes/CU_Work/senior_project/exam_phase/data/raw/screen staring/gritt_exam.mov"

In [21]:
detect_mouth_state_behavior(SUBJECT_VIDEO_PATH)

Detecting mouth state behavior...


: 